# Wire an Auto Scaling Group to an Application Load Balancer and watch a scaling event execute end-to-end

A founder runs a marketing site on a single t4g.nano with a static Elastic IP. Every time a Hacker News post drives traffic, the box hits CPU and starts dropping requests. The founder asks you to build the smallest production-shaped autoscaling pattern that absorbs a 10x spike without paying for headroom 24/7.

You will build:

- A launch template that pins a t4g.nano instance type and an Amazon Linux 2023 ARM AMI (resolved at create time via SSM).
- A target group with HTTP health checks on `/`.
- An Application Load Balancer across two AZs forwarding to the target group.
- An Auto Scaling Group (min 1, max 3, desired 1) across the same two AZs, attached to the target group.
- A target-tracking scaling policy on `ALBRequestCountPerTarget=50`.

Then you drive synthetic load with a helper, observe the ASG's desired count move from 1 to 2 (or 3) within the cooldown window, and read the most recent scaling activity from `describe_scaling_activities` to see the CloudWatch-alarm-fire human-readable cause.

**Time.** Plan for about 60 minutes hands-on. Most of the wait time is in two places: a 2-3 minute window for the first ASG instance to pass the target group health check, and a 3-5 minute window for the synthetic load + scaling activity to land.

**Cost.** This lab costs about a nickel if you clean up promptly. The ALB is the line item that matters at $0.0225/hour, plus the EC2 instances at less than half a cent per hour each. The cost guardrails matter most if you walk away with the ASG still running: 1 ALB + 3 instances overnight is about 65 cents, a week of forgotten infrastructure is about $4.50. The companion panel will start poking you at 30 minutes of kernel idle.

This lab maps to AWS SAA-C03 Domain 2 (Design Resilient Architectures, 26%).

In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned per LAB-CREATION-BLUEPRINT.md build rules.

!pip install --quiet labrun-checks==0.6.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants.

import atexit
import base64
import datetime as dt
import getpass
import time
from concurrent.futures import ThreadPoolExecutor
from urllib.error import URLError
from urllib.request import urlopen

import boto3
from botocore.exceptions import ClientError
from IPython.display import clear_output

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
)

LAB_ID = "auto-scaling-with-alb"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"

LAUNCH_TEMPLATE_NAME = f"labrun-{LAB_ID}-lt"
SG_INSTANCE_NAME = f"labrun-{LAB_ID}-sg"
SG_ALB_NAME = f"labrun-{LAB_ID}-alb-sg"
TARGET_GROUP_NAME = f"labrun-{LAB_ID}-tg"
ALB_NAME = f"labrun-{LAB_ID}-alb"
ASG_NAME = f"labrun-{LAB_ID}-asg"
SCALING_POLICY_NAME = f"labrun-{LAB_ID}-policy"
EC2_ROLE_NAME = f"labrun-{LAB_ID}-ec2-role"
INSTANCE_PROFILE_NAME = f"labrun-{LAB_ID}-instance-profile"

LAB_STATE = {
    "session_start": None,
    "vpc_id": None,
    "subnet_ids": [],
    "azs": [],
    "ami_id": None,
    "launch_template_id": None,
    "instance_sg_id": None,
    "alb_sg_id": None,
    "target_group_arn": None,
    "alb_arn": None,
    "alb_dns_name": None,
    "listener_arn": None,
    "asg_created": False,
    "scaling_policy_arn": None,
    "scale_out_desired": None,
}

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials, and resolve the AL2023
# ARM AMI from SSM.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Region: {REGION}")

# Resolve AL2023 ARM AMI for the launch template (validators re-resolve).
ssm = boto3.client(
    "ssm",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
LAB_STATE["ami_id"] = ssm.get_parameter(
    Name="/aws/service/ami-amazon-linux-latest/al2023-ami-kernel-default-arm64",
)["Parameter"]["Value"]
print(f"AL2023 ARM AMI: {LAB_STATE['ami_id']}")

# Pick the default VPC and two AZ-distinct subnets.
ec2 = boto3.client(
    "ec2",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
default_vpcs = ec2.describe_vpcs(Filters=[{"Name": "isDefault", "Values": ["true"]}])["Vpcs"]
if not default_vpcs:
    print("No default VPC. Lab 6 expects a default VPC.")
    raise SystemExit(1)
LAB_STATE["vpc_id"] = default_vpcs[0]["VpcId"]
subnets = ec2.describe_subnets(
    Filters=[{"Name": "vpc-id", "Values": [LAB_STATE["vpc_id"]]}],
)["Subnets"]
azs_seen = {}
for sn in subnets:
    azs_seen.setdefault(sn["AvailabilityZone"], sn["SubnetId"])
azs_sorted = sorted(azs_seen.keys())[:2]
LAB_STATE["azs"] = azs_sorted
LAB_STATE["subnet_ids"] = [azs_seen[az] for az in azs_sorted]
print(f"Default VPC: {LAB_STATE['vpc_id']}")
print(f"AZs:    {LAB_STATE['azs']}")
print(f"Subnets: {LAB_STATE['subnet_ids']}")

LAB_STATE["session_start"] = dt.datetime.now(dt.timezone.utc)
register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan. Critical-first
# ordering: ASG scale-to-zero first (waits up to 5 minutes for instances to
# terminate), then ASG delete, listener, ALB (waits for delete), target
# group, launch template, security groups (retries on DependencyViolation),
# instance profile, IAM role.

CLEANUP_MANIFEST = []


def _rebuild_manifest():
    CLEANUP_MANIFEST.clear()
    if LAB_STATE["asg_created"]:
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="autoscaling_group_scale_to_zero",
                id=ASG_NAME,
                region=REGION,
                cli_delete_command=(
                    f"aws autoscaling update-auto-scaling-group "
                    f"--auto-scaling-group-name {ASG_NAME} "
                    f"--min-size 0 --max-size 0 --desired-capacity 0"
                ),
            )
        )
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="autoscaling_group",
                id=ASG_NAME,
                region=REGION,
                cli_delete_command=(
                    f"aws autoscaling delete-auto-scaling-group "
                    f"--auto-scaling-group-name {ASG_NAME} --force-delete"
                ),
            )
        )
    if LAB_STATE["listener_arn"]:
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="elbv2_listener",
                id=LAB_STATE["listener_arn"],
                region=REGION,
                cli_delete_command=(
                    f"aws elbv2 delete-listener --listener-arn {LAB_STATE['listener_arn']}"
                ),
            )
        )
    if LAB_STATE["alb_arn"]:
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="elbv2_load_balancer",
                id=LAB_STATE["alb_arn"],
                region=REGION,
                cli_delete_command=(
                    f"aws elbv2 delete-load-balancer --load-balancer-arn {LAB_STATE['alb_arn']}"
                ),
            )
        )
    if LAB_STATE["target_group_arn"]:
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="elbv2_target_group",
                id=LAB_STATE["target_group_arn"],
                region=REGION,
                cli_delete_command=(
                    f"aws elbv2 delete-target-group "
                    f"--target-group-arn {LAB_STATE['target_group_arn']}"
                ),
            )
        )
    if LAB_STATE["launch_template_id"]:
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="ec2_launch_template",
                id=LAB_STATE["launch_template_id"],
                region=REGION,
                cli_delete_command=(
                    f"aws ec2 delete-launch-template "
                    f"--launch-template-id {LAB_STATE['launch_template_id']}"
                ),
            )
        )
    for sg_id in (LAB_STATE["instance_sg_id"], LAB_STATE["alb_sg_id"]):
        if sg_id:
            CLEANUP_MANIFEST.append(
                CleanupResource(
                    type="ec2_security_group",
                    id=sg_id,
                    region=REGION,
                    cli_delete_command=f"aws ec2 delete-security-group --group-id {sg_id}",
                )
            )
    CLEANUP_MANIFEST.append(
        CleanupResource(
            type="iam_instance_profile",
            id=INSTANCE_PROFILE_NAME,
            parent=EC2_ROLE_NAME,
            region=REGION,
            cli_delete_command=(
                f"aws iam remove-role-from-instance-profile "
                f"--instance-profile-name {INSTANCE_PROFILE_NAME} "
                f"--role-name {EC2_ROLE_NAME} && "
                f"aws iam delete-instance-profile "
                f"--instance-profile-name {INSTANCE_PROFILE_NAME}"
            ),
        )
    )
    CLEANUP_MANIFEST.append(
        CleanupResource(
            type="iam_role",
            id=EC2_ROLE_NAME,
            region=REGION,
            cli_delete_command=f"aws iam delete-role --role-name {EC2_ROLE_NAME}",
        )
    )


_rebuild_manifest()


def _atexit_cleanup():
    try:
        _rebuild_manifest()
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans():
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom of this notebook first, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to build the ASG + ALB stack.")

## Task 1: Author the launch template

The launch template captures the immutable shape of each ASG-managed instance: AMI, instance type, instance profile, security group, and user-data. The ASG reads it at scale-out time.

Build the IAM role + instance profile first (the instance profile is empty IAM-wise; the user-data does not call AWS APIs, but the launch template requires an instance profile reference to launch). Then create the launch template with the data below.

The user-data is a one-shot HTTP server: `python3 -m http.server 80` serves a 200-byte response on `/`. AL2023 ships with Python 3 pre-installed.

In [ ]:
# NBVAL_SKIP
# Task 1: IAM role + instance profile + launch template.

import json

iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
ec2 = boto3.client(
    "ec2",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Trust policy for EC2.
trust = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "ec2.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}
try:
    iam.create_role(
        RoleName=EC2_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust),
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
except ClientError as e:
    if e.response.get("Error", {}).get("Code") != "EntityAlreadyExists":
        raise
try:
    iam.create_instance_profile(
        InstanceProfileName=INSTANCE_PROFILE_NAME,
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
except ClientError as e:
    if e.response.get("Error", {}).get("Code") != "EntityAlreadyExists":
        raise
try:
    iam.add_role_to_instance_profile(
        InstanceProfileName=INSTANCE_PROFILE_NAME, RoleName=EC2_ROLE_NAME
    )
except ClientError as e:
    if e.response.get("Error", {}).get("Code") != "LimitExceeded":
        raise

# Wait for IAM eventual consistency before EC2 sees the instance profile.
print("Waiting 15s for IAM instance profile to propagate...")
time.sleep(15)

# User-data: serve a 200-byte response on port 80.
user_data_script = (
    "#!/bin/bash\n"
    "set -e\n"
    "mkdir -p /var/lib/labrun\n"
    "cat > /var/lib/labrun/index.html <<'BODY'\n"
    "<html><body>labrun ASG instance OK. Hostname: $(hostname). "
    "Serving from /var/lib/labrun on port 80.</body></html>\n"
    "BODY\n"
    "cd /var/lib/labrun\n"
    "nohup python3 -m http.server 80 > /var/log/labrun.log 2>&1 &\n"
)
user_data = base64.b64encode(user_data_script.encode("utf-8")).decode("utf-8")

# Create the launch template.
# YOUR CODE: call ec2.create_launch_template with LaunchTemplateName=LAUNCH_TEMPLATE_NAME,
# TagSpecifications=[{"ResourceType": "launch-template", "Tags": [{"Key": LAB_TAG_KEY,
# "Value": LAB_TAG_VALUE}]}], and LaunchTemplateData={
#     "ImageId": LAB_STATE["ami_id"],
#     "InstanceType": "t4g.nano",
#     "IamInstanceProfile": {"Name": INSTANCE_PROFILE_NAME},
#     "UserData": user_data,
#     "TagSpecifications": [{
#         "ResourceType": "instance",
#         "Tags": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE},
#                  {"Key": "Name", "Value": f"labrun-{LAB_ID}-instance"}],
#     }],
# }. Capture LaunchTemplateId in LAB_STATE["launch_template_id"]. Handle
# InvalidLaunchTemplateName.AlreadyExistsException by reading the existing
# template's ID with describe_launch_templates.

_rebuild_manifest()
print(f"Launch template: {LAB_STATE['launch_template_id']}")

In [ ]:
# NBVAL_SKIP
# Observe: confirm the launch template exists and its data shape is right.
# Single snapshot, not a poll loop.

ec2_obs = boto3.client(
    "ec2",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

clear_output(wait=True)
try:
    tmpl = ec2_obs.describe_launch_template_versions(
        LaunchTemplateId=LAB_STATE["launch_template_id"], Versions=["$Latest"],
    )["LaunchTemplateVersions"][0]
    data = tmpl["LaunchTemplateData"]
except (ClientError, IndexError) as e:
    print(f"Error reading launch template: {e}")
    data = {}

print("Launch template ($Latest version) summary:")
print("-" * 64)
print(f"  ID:             {LAB_STATE['launch_template_id']}")
print(f"  InstanceType:   {data.get('InstanceType')}")
print(f"  ImageId:        {data.get('ImageId')}")
print(f"  InstanceProfile:{(data.get('IamInstanceProfile') or {}).get('Name')}")
ud = data.get("UserData", "")
if ud:
    try:
        decoded = base64.b64decode(ud).decode("utf-8")
        contains_http_server = "python3 -m http.server 80" in decoded
    except Exception:
        contains_http_server = False
    print(f"  HTTP server in user-data: {contains_http_server}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Launch template exists with the right shape.

def checkpoint_1(session):
    from labrun_checks import CheckpointResult
    try:
        ec2c = boto3.client(
            "ec2",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        tmpls = ec2c.describe_launch_templates(
            Filters=[{"Name": "tag:" + LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
        ).get("LaunchTemplates", [])
        if len(tmpls) != 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Expected 1 launch template tagged for this lab, found {len(tmpls)}."
                ),
            )
        ver = ec2c.describe_launch_template_versions(
            LaunchTemplateId=tmpls[0]["LaunchTemplateId"], Versions=["$Latest"],
        )["LaunchTemplateVersions"][0]
        data = ver.get("LaunchTemplateData", {})
        if data.get("InstanceType") != "t4g.nano":
            return CheckpointResult(
                status="fail",
                error_reason=f"InstanceType is {data.get('InstanceType')!r}, expected t4g.nano.",
            )
        if not data.get("ImageId", "").startswith("ami-"):
            return CheckpointResult(
                status="fail",
                error_reason="ImageId is missing or not a valid AMI reference.",
            )
        prof = (data.get("IamInstanceProfile") or {}).get("Name")
        if prof != INSTANCE_PROFILE_NAME:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"IamInstanceProfile.Name is {prof!r}, expected {INSTANCE_PROFILE_NAME!r}."
                ),
            )
        ud_b64 = data.get("UserData", "")
        if not ud_b64:
            return CheckpointResult(
                status="fail",
                error_reason="UserData is empty. The launch template needs the HTTP server boot script.",
            )
        try:
            decoded = base64.b64decode(ud_b64).decode("utf-8")
        except Exception:
            decoded = ""
        if "python3 -m http.server 80" not in decoded:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "UserData does not contain `python3 -m http.server 80`. The "
                    "HTTP server boot is what passes the ALB health check."
                ),
            )
        return CheckpointResult(status="pass")
    except ClientError as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

`create_launch_template` takes a name, a tag-spec for the template itself, and a `LaunchTemplateData` dict with the instance shape. The ASG reads `$Latest` of the template at scale-out, so versioning is automatic.

</details>

<details><summary>Hint 2 (direction)</summary>

The four fields that matter: `ImageId` (the AL2023 ARM AMI you resolved in setup), `InstanceType="t4g.nano"`, `IamInstanceProfile={"Name": INSTANCE_PROFILE_NAME}` (use Name not Arn so AWS resolves the latest), and `UserData` (already prepared as a base64 string). Add an instance-level tag-spec inside `LaunchTemplateData` so launched instances inherit the lab tag.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
try:
    resp = ec2.create_launch_template(
        LaunchTemplateName=LAUNCH_TEMPLATE_NAME,
        TagSpecifications=[
            {
                "ResourceType": "launch-template",
                "Tags": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
            }
        ],
        LaunchTemplateData={
            "ImageId": LAB_STATE["ami_id"],
            "InstanceType": "t4g.nano",
            "IamInstanceProfile": {"Name": INSTANCE_PROFILE_NAME},
            "UserData": user_data,
            "TagSpecifications": [
                {
                    "ResourceType": "instance",
                    "Tags": [
                        {"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE},
                        {"Key": "Name", "Value": f"labrun-{LAB_ID}-instance"},
                    ],
                }
            ],
        },
    )
    LAB_STATE["launch_template_id"] = resp["LaunchTemplate"]["LaunchTemplateId"]
except ClientError as e:
    if e.response.get("Error", {}).get("Code") == "InvalidLaunchTemplateName.AlreadyExistsException":
        existing = ec2.describe_launch_templates(LaunchTemplateNames=[LAUNCH_TEMPLATE_NAME])
        LAB_STATE["launch_template_id"] = existing["LaunchTemplates"][0]["LaunchTemplateId"]
    else:
        raise
```

</details>

**Wallet check.** Still at $0.00. Launch templates and IAM are always free; the ALB and EC2 instances do not exist yet.

## Task 2: Build the target group, ALB, and HTTP listener

Three boto3 calls, in order:

1. `elbv2.create_target_group(Protocol="HTTP", Port=80, TargetType="instance", VpcId=..., HealthCheckPath="/", HealthCheckIntervalSeconds=15, HealthyThresholdCount=2)`.
2. Two security groups: one for the ALB (inbound 80 from 0.0.0.0/0), one for the instances (inbound 80 from the ALB SG).
3. `elbv2.create_load_balancer(Name=ALB_NAME, Subnets=..., SecurityGroups=[ALB_SG], Scheme="internet-facing", Type="application")`.
4. `elbv2.create_listener(LoadBalancerArn=..., Protocol="HTTP", Port=80, DefaultActions=[{"Type": "forward", "TargetGroupArn": ...}])`.

Tag each one at creation. The observe cell polls the ALB until `State.Code=active` (takes about 60-90 seconds).

In [ ]:
# NBVAL_SKIP
# Task 2: security groups, target group, ALB, listener.

ec2 = boto3.client(
    "ec2",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
elbv2 = boto3.client(
    "elbv2",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# ALB security group: inbound 80 from anywhere.
try:
    alb_sg = ec2.create_security_group(
        GroupName=SG_ALB_NAME,
        Description="labrun ALB SG",
        VpcId=LAB_STATE["vpc_id"],
        TagSpecifications=[
            {
                "ResourceType": "security-group",
                "Tags": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
            }
        ],
    )
    LAB_STATE["alb_sg_id"] = alb_sg["GroupId"]
    ec2.authorize_security_group_ingress(
        GroupId=LAB_STATE["alb_sg_id"],
        IpPermissions=[
            {
                "IpProtocol": "tcp",
                "FromPort": 80,
                "ToPort": 80,
                "IpRanges": [{"CidrIp": "0.0.0.0/0"}],
            }
        ],
    )
except ClientError as e:
    if e.response.get("Error", {}).get("Code") == "InvalidGroup.Duplicate":
        sgs = ec2.describe_security_groups(
            Filters=[{"Name": "group-name", "Values": [SG_ALB_NAME]}],
        )["SecurityGroups"]
        LAB_STATE["alb_sg_id"] = sgs[0]["GroupId"]
    else:
        raise

# Instance security group: inbound 80 from the ALB SG.
try:
    inst_sg = ec2.create_security_group(
        GroupName=SG_INSTANCE_NAME,
        Description="labrun instance SG",
        VpcId=LAB_STATE["vpc_id"],
        TagSpecifications=[
            {
                "ResourceType": "security-group",
                "Tags": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
            }
        ],
    )
    LAB_STATE["instance_sg_id"] = inst_sg["GroupId"]
    ec2.authorize_security_group_ingress(
        GroupId=LAB_STATE["instance_sg_id"],
        IpPermissions=[
            {
                "IpProtocol": "tcp",
                "FromPort": 80,
                "ToPort": 80,
                "UserIdGroupPairs": [{"GroupId": LAB_STATE["alb_sg_id"]}],
            }
        ],
    )
except ClientError as e:
    if e.response.get("Error", {}).get("Code") == "InvalidGroup.Duplicate":
        sgs = ec2.describe_security_groups(
            Filters=[{"Name": "group-name", "Values": [SG_INSTANCE_NAME]}],
        )["SecurityGroups"]
        LAB_STATE["instance_sg_id"] = sgs[0]["GroupId"]
    else:
        raise

# Update the launch template to reference the instance SG. ASG launches read
# SecurityGroupIds from the template's latest version.
ec2.create_launch_template_version(
    LaunchTemplateId=LAB_STATE["launch_template_id"],
    SourceVersion="$Latest",
    LaunchTemplateData={"SecurityGroupIds": [LAB_STATE["instance_sg_id"]]},
)
ec2.modify_launch_template(
    LaunchTemplateId=LAB_STATE["launch_template_id"], DefaultVersion="$Latest"
)

# Target group.
# YOUR CODE: call elbv2.create_target_group with Name=TARGET_GROUP_NAME,
# Protocol="HTTP", Port=80, VpcId=LAB_STATE["vpc_id"], TargetType="instance",
# HealthCheckProtocol="HTTP", HealthCheckPath="/", HealthCheckIntervalSeconds=15,
# HealthyThresholdCount=2, Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
# Capture TargetGroupArn in LAB_STATE["target_group_arn"]. Handle
# DuplicateTargetGroupName as a no-op (re-describe by name).

# ALB.
# YOUR CODE: call elbv2.create_load_balancer with Name=ALB_NAME,
# Subnets=LAB_STATE["subnet_ids"], SecurityGroups=[LAB_STATE["alb_sg_id"]],
# Scheme="internet-facing", Type="application", IpAddressType="ipv4",
# Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]. Capture LoadBalancerArn
# in LAB_STATE["alb_arn"] and DNSName in LAB_STATE["alb_dns_name"]. Handle
# DuplicateLoadBalancerName as a no-op.

# Listener.
# YOUR CODE: call elbv2.create_listener with LoadBalancerArn=LAB_STATE["alb_arn"],
# Protocol="HTTP", Port=80, DefaultActions=[{"Type": "forward", "TargetGroupArn":
# LAB_STATE["target_group_arn"]}], Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
# Capture ListenerArn in LAB_STATE["listener_arn"]. Handle DuplicateListener
# as a no-op (re-describe by ALB ARN).

_rebuild_manifest()
print(f"ALB SG:          {LAB_STATE['alb_sg_id']}")
print(f"Instance SG:     {LAB_STATE['instance_sg_id']}")
print(f"Target group:    {LAB_STATE['target_group_arn']}")
print(f"ALB:             {LAB_STATE['alb_arn']}")
print(f"ALB DNS:         {LAB_STATE['alb_dns_name']}")
print(f"Listener:        {LAB_STATE['listener_arn']}")

In [ ]:
# NBVAL_SKIP
# Observe: poll the ALB until State.Code=active. Takes ~60-90 seconds.

elbv2_obs = boto3.client(
    "elbv2",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

deadline = time.time() + 240
while time.time() < deadline:
    clear_output(wait=True)
    try:
        alb = elbv2_obs.describe_load_balancers(
            LoadBalancerArns=[LAB_STATE["alb_arn"]],
        )["LoadBalancers"][0]
        state = alb.get("State", {}).get("Code", "?")
        dns = alb.get("DNSName", "?")
        azs = sorted([z["ZoneName"] for z in alb.get("AvailabilityZones", [])])
    except (ClientError, IndexError) as e:
        state = f"error: {e}"
        dns = "?"
        azs = []
    elapsed = int(240 - (deadline - time.time()))
    print(f"ALB state at t+{elapsed:>3}s:")
    print("-" * 64)
    print(f"  ALB:           {LAB_STATE['alb_arn']}")
    print(f"  State:         {state}")
    print(f"  DNS:           {dns}")
    print(f"  AZs:           {azs}")
    if state == "active":
        print()
        print("ALB is active. Listener is forwarding to the target group.")
        break
    time.sleep(10)
else:
    print()
    print("Timed out waiting for ALB to become active.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: target group + ALB + listener with the expected shape.

def checkpoint_2(session):
    from labrun_checks import CheckpointResult
    try:
        elbv2_c = boto3.client(
            "elbv2",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        tgs = elbv2_c.describe_target_groups(Names=[TARGET_GROUP_NAME])["TargetGroups"]
        tg = tgs[0]
        if tg.get("Protocol") != "HTTP" or tg.get("Port") != 80:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Target group protocol/port is {tg.get('Protocol')}/{tg.get('Port')}, "
                    f"expected HTTP/80."
                ),
            )
        if tg.get("HealthCheckPath") != "/":
            return CheckpointResult(
                status="fail",
                error_reason=f"HealthCheckPath is {tg.get('HealthCheckPath')!r}, expected '/'.",
            )
        if tg.get("HealthCheckIntervalSeconds") != 15:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"HealthCheckIntervalSeconds is {tg.get('HealthCheckIntervalSeconds')}, "
                    f"expected 15."
                ),
            )
        if tg.get("HealthyThresholdCount") != 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"HealthyThresholdCount is {tg.get('HealthyThresholdCount')}, expected 2."
                ),
            )
        if tg.get("TargetType") != "instance":
            return CheckpointResult(
                status="fail",
                error_reason=f"TargetType is {tg.get('TargetType')!r}, expected 'instance'.",
            )

        alb = elbv2_c.describe_load_balancers(Names=[ALB_NAME])["LoadBalancers"][0]
        if alb.get("State", {}).get("Code") != "active":
            return CheckpointResult(
                status="fail",
                error_reason=f"ALB state is {alb.get('State', {}).get('Code')!r}, expected active.",
            )
        if alb.get("Type") != "application":
            return CheckpointResult(
                status="fail",
                error_reason=f"ALB type is {alb.get('Type')!r}, expected application.",
            )
        if len(alb.get("AvailabilityZones", [])) != 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"ALB spans {len(alb.get('AvailabilityZones', []))} AZs, expected 2."
                ),
            )

        listeners = elbv2_c.describe_listeners(LoadBalancerArn=alb["LoadBalancerArn"])
        if not listeners.get("Listeners"):
            return CheckpointResult(
                status="fail",
                error_reason="ALB has no listeners. Add one forwarding to the target group.",
            )
        listener = listeners["Listeners"][0]
        if listener.get("Port") != 80 or listener.get("Protocol") != "HTTP":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Listener is {listener.get('Protocol')}/{listener.get('Port')}, expected HTTP/80."
                ),
            )
        actions = listener.get("DefaultActions", [])
        if not actions or actions[0].get("TargetGroupArn") != tg["TargetGroupArn"]:
            return CheckpointResult(
                status="fail",
                error_reason="Listener does not forward to the lab target group.",
            )
        return CheckpointResult(status="pass")
    except (ClientError, IndexError, KeyError) as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

Three ELBv2 calls in order: target group, ALB, listener. The listener references both the ALB and the target group, so the order matters.

</details>

<details><summary>Hint 2 (direction)</summary>

Target group: pin `HealthCheckIntervalSeconds=15` and `HealthyThresholdCount=2` so a t4g.nano with a Python HTTP server passes the check in ~30-45 seconds after launch. ALB: pass the two subnet IDs in `LAB_STATE["subnet_ids"]` and the ALB security group. Listener: `DefaultActions=[{"Type": "forward", "TargetGroupArn": ...}]`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
tg = elbv2.create_target_group(
    Name=TARGET_GROUP_NAME,
    Protocol="HTTP",
    Port=80,
    VpcId=LAB_STATE["vpc_id"],
    TargetType="instance",
    HealthCheckProtocol="HTTP",
    HealthCheckPath="/",
    HealthCheckIntervalSeconds=15,
    HealthyThresholdCount=2,
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)["TargetGroups"][0]
LAB_STATE["target_group_arn"] = tg["TargetGroupArn"]

alb = elbv2.create_load_balancer(
    Name=ALB_NAME,
    Subnets=LAB_STATE["subnet_ids"],
    SecurityGroups=[LAB_STATE["alb_sg_id"]],
    Scheme="internet-facing",
    Type="application",
    IpAddressType="ipv4",
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)["LoadBalancers"][0]
LAB_STATE["alb_arn"] = alb["LoadBalancerArn"]
LAB_STATE["alb_dns_name"] = alb["DNSName"]

listener = elbv2.create_listener(
    LoadBalancerArn=LAB_STATE["alb_arn"],
    Protocol="HTTP",
    Port=80,
    DefaultActions=[{"Type": "forward", "TargetGroupArn": LAB_STATE["target_group_arn"]}],
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)["Listeners"][0]
LAB_STATE["listener_arn"] = listener["ListenerArn"]
```

</details>

**Wallet check.** ALB just started billing at $0.0225/hour. About 0.04 cents per minute. The 60-minute session will land around 2.5 cents on ALB hours alone.

## Task 3: Auto Scaling Group + target-tracking scaling policy

The ASG is the bridge between the launch template and the target group. Min 1, max 3, desired 1; AZs match the ALB's two AZs; target group attached so launched instances register automatically.

After the ASG is created, attach a target-tracking scaling policy on `ALBRequestCountPerTarget` with a target value of 50 requests per minute per target. AWS auto-creates the CloudWatch alarms behind the scenes; you do not write the alarm yourself.

The observe cell polls until the ASG's first instance reaches `healthy` in the target group (2-4 minutes typical, ceiling 5).

In [ ]:
# NBVAL_SKIP
# Task 3: Auto Scaling Group + target-tracking policy.

asg = boto3.client(
    "autoscaling",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Create the ASG.
# YOUR CODE: call asg.create_auto_scaling_group with AutoScalingGroupName=ASG_NAME,
# LaunchTemplate={"LaunchTemplateId": LAB_STATE["launch_template_id"], "Version": "$Latest"},
# MinSize=1, MaxSize=3, DesiredCapacity=1,
# VPCZoneIdentifier=",".join(LAB_STATE["subnet_ids"]),
# TargetGroupARNs=[LAB_STATE["target_group_arn"]],
# HealthCheckType="ELB", HealthCheckGracePeriod=120,
# Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE, "PropagateAtLaunch": True,
# "ResourceId": ASG_NAME, "ResourceType": "auto-scaling-group"}]. Handle
# AlreadyExistsFault as a no-op.

LAB_STATE["asg_created"] = True

# Attach target-tracking scaling policy.
# YOUR CODE: call asg.put_scaling_policy with AutoScalingGroupName=ASG_NAME,
# PolicyName=SCALING_POLICY_NAME, PolicyType="TargetTrackingScaling",
# TargetTrackingConfiguration={
#     "PredefinedMetricSpecification": {
#         "PredefinedMetricType": "ALBRequestCountPerTarget",
#         "ResourceLabel": <see hint below>,
#     },
#     "TargetValue": 50.0,
# }. Capture PolicyARN in LAB_STATE["scaling_policy_arn"].
#
# ResourceLabel format is `app/<alb-name>/<alb-id>/targetgroup/<tg-name>/<tg-id>`;
# build it from LAB_STATE["alb_arn"] and LAB_STATE["target_group_arn"] (each
# ARN's last segment is the id; the suffix is everything after the account id).

_rebuild_manifest()
print(f"ASG:               {ASG_NAME}")
print(f"Scaling policy:    {LAB_STATE['scaling_policy_arn']}")
print()
print("Run the observe cell to watch the first instance launch and pass the health check.")

In [ ]:
# NBVAL_SKIP
# Observe: poll until at least one target in the target group is healthy.
# Ceiling 5 minutes (instance launch + status checks + ALB health check).

elbv2_obs = boto3.client(
    "elbv2",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
asg_obs = boto3.client(
    "autoscaling",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

deadline = time.time() + 360
while time.time() < deadline:
    clear_output(wait=True)
    try:
        health = elbv2_obs.describe_target_health(
            TargetGroupArn=LAB_STATE["target_group_arn"],
        )["TargetHealthDescriptions"]
    except ClientError as e:
        health = [{"Target": {"Id": str(e)}, "TargetHealth": {"State": "error"}}]
    try:
        groups = asg_obs.describe_auto_scaling_groups(
            AutoScalingGroupNames=[ASG_NAME],
        )["AutoScalingGroups"][0]
        desired = groups.get("DesiredCapacity")
    except (ClientError, IndexError):
        desired = "?"
    elapsed = int(360 - (deadline - time.time()))
    print(f"ASG + target health at t+{elapsed:>3}s:")
    print("-" * 64)
    print(f"  ASG DesiredCapacity: {desired}")
    print(f"  Target group:        {LAB_STATE['target_group_arn']}")
    print(f"  Targets:")
    for h in health:
        t_id = (h.get("Target") or {}).get("Id", "?")
        t_state = (h.get("TargetHealth") or {}).get("State", "?")
        t_reason = (h.get("TargetHealth") or {}).get("Reason", "")
        print(f"    {t_id:20}  state={t_state:12}  reason={t_reason}")
    healthy = [h for h in health if (h.get("TargetHealth") or {}).get("State") == "healthy"]
    if healthy:
        print()
        print(f"Health check passing. ASG instance is registered and serving traffic.")
        break
    time.sleep(15)
else:
    print()
    print("Timed out waiting for a healthy target. Likely causes:")
    print("  - User-data did not start the HTTP server (check EC2 console system log).")
    print("  - Instance SG does not allow inbound 80 from the ALB SG.")
    print("  - Health check path is not '/' on the target group.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: ASG exists with the expected shape.

def checkpoint_3(session):
    from labrun_checks import CheckpointResult
    try:
        asg_c = boto3.client(
            "autoscaling",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        groups = asg_c.describe_auto_scaling_groups(
            AutoScalingGroupNames=[ASG_NAME],
        )["AutoScalingGroups"]
        if not groups:
            return CheckpointResult(
                status="fail",
                error_reason=f"ASG {ASG_NAME!r} does not exist.",
            )
        g = groups[0]
        if g.get("MinSize") != 1 or g.get("MaxSize") != 3 or g.get("DesiredCapacity") not in (1, 2, 3):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"ASG sizing is Min={g.get('MinSize')} Max={g.get('MaxSize')} "
                    f"Desired={g.get('DesiredCapacity')}, expected Min=1 Max=3 Desired in (1,2,3)."
                ),
            )
        if len(g.get("AvailabilityZones", [])) != 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"ASG spans {len(g.get('AvailabilityZones', []))} AZs, expected 2."
                ),
            )
        if LAB_STATE["target_group_arn"] not in g.get("TargetGroupARNs", []):
            return CheckpointResult(
                status="fail",
                error_reason="ASG is not attached to the lab target group.",
            )
        if (g.get("LaunchTemplate") or {}).get("LaunchTemplateId") != LAB_STATE["launch_template_id"]:
            return CheckpointResult(
                status="fail",
                error_reason="ASG is not pointed at the lab launch template.",
            )
        return CheckpointResult(status="pass")
    except ClientError as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

Two `autoscaling` calls. `create_auto_scaling_group` builds the group; `put_scaling_policy` attaches the target-tracking policy. The target-tracking policy auto-creates two CloudWatch alarms behind the scenes for you.

</details>

<details><summary>Hint 2 (direction)</summary>

`VPCZoneIdentifier` is a comma-separated string of subnet IDs, not a list. `LaunchTemplate={"LaunchTemplateId": ..., "Version": "$Latest"}` so the ASG reads the latest version (the one that has the SG attached). Tag the ASG via the `Tags` argument with `PropagateAtLaunch=True` so each launched instance inherits the tag.

For the scaling policy, the `ResourceLabel` is the target-tracking-specific way to point at the ALB-and-target-group pair. Format: `app/<alb-name>/<alb-hex-id>/targetgroup/<tg-name>/<tg-hex-id>`. Build it from the ARN suffixes.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
try:
    asg.create_auto_scaling_group(
        AutoScalingGroupName=ASG_NAME,
        LaunchTemplate={"LaunchTemplateId": LAB_STATE["launch_template_id"], "Version": "$Latest"},
        MinSize=1,
        MaxSize=3,
        DesiredCapacity=1,
        VPCZoneIdentifier=",".join(LAB_STATE["subnet_ids"]),
        TargetGroupARNs=[LAB_STATE["target_group_arn"]],
        HealthCheckType="ELB",
        HealthCheckGracePeriod=120,
        Tags=[
            {
                "ResourceId": ASG_NAME,
                "ResourceType": "auto-scaling-group",
                "Key": LAB_TAG_KEY,
                "Value": LAB_TAG_VALUE,
                "PropagateAtLaunch": True,
            }
        ],
    )
except ClientError as e:
    if e.response.get("Error", {}).get("Code") != "AlreadyExists":
        raise

alb_suffix = LAB_STATE["alb_arn"].split(":loadbalancer/")[1]
tg_suffix = "targetgroup/" + LAB_STATE["target_group_arn"].split(":targetgroup/")[1]
resource_label = f"{alb_suffix}/{tg_suffix}"

resp = asg.put_scaling_policy(
    AutoScalingGroupName=ASG_NAME,
    PolicyName=SCALING_POLICY_NAME,
    PolicyType="TargetTrackingScaling",
    TargetTrackingConfiguration={
        "PredefinedMetricSpecification": {
            "PredefinedMetricType": "ALBRequestCountPerTarget",
            "ResourceLabel": resource_label,
        },
        "TargetValue": 50.0,
    },
)
LAB_STATE["scaling_policy_arn"] = resp["PolicyARN"]
```

</details>

**Wallet check.** The first ASG instance has been running about 4 minutes by now: 0.03 cents. The ALB and target group are doing the bulk of the bill: about 1.5 cents combined. Running total roughly 3 cents.

## Task 4: Drive synthetic load and watch the ASG scale out

The synthetic load helper issues HTTP GETs against `LAB_STATE["alb_dns_name"]` in parallel for 3 minutes. The `ALBRequestCountPerTarget` metric publishes every minute; once the rolling per-target rate crosses 50, the target-tracking policy fires and the ASG bumps `DesiredCapacity` from 1 to 2.

The observe cell polls both `DesiredCapacity` and the most recent scaling activity. The scaling activity's `Cause` field gives you the CloudWatch-alarm-fire reason in human-readable text (something like `"At 2026-05-13T... a monitor alarm... in state ALARM..."`). Ceiling 6 minutes after the load helper starts.

In [ ]:
# NBVAL_SKIP
# Task 4: drive synthetic load, watch the ASG scale out.

def synthetic_load(duration_seconds=180, concurrency=20):
    """Hit the ALB DNS with parallel GETs for `duration_seconds`. The
    helper sets a low socket timeout so failed-to-connect responses cycle
    fast and the metric still climbs above the 50 RCPT threshold."""
    url = f"http://{LAB_STATE['alb_dns_name']}/"
    deadline = time.time() + duration_seconds
    counts = {"ok": 0, "err": 0}

    def hit():
        try:
            with urlopen(url, timeout=2) as resp:
                resp.read(64)
                counts["ok"] += 1
        except (URLError, TimeoutError, Exception):
            counts["err"] += 1

    with ThreadPoolExecutor(max_workers=concurrency) as pool:
        while time.time() < deadline:
            pool.submit(hit)
            time.sleep(0.05)
    return counts


print("Driving synthetic load against the ALB for 3 minutes...")
print(f"  URL: http://{LAB_STATE['alb_dns_name']}/")
results = synthetic_load()
print(f"Load helper finished. Requests: ok={results['ok']}, err={results['err']}")
print()
print("Now run the observe cell to watch the ASG DesiredCapacity climb.")

In [ ]:
# NBVAL_SKIP
# Observe: poll the ASG every 15 seconds until DesiredCapacity >= 2, with a
# 6-minute ceiling. Surface the most recent scaling activity Cause field
# from describe_scaling_activities so the alarm-fire reason is visible.

asg_obs = boto3.client(
    "autoscaling",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

deadline = time.time() + 360
final_desired = 1
while time.time() < deadline:
    clear_output(wait=True)
    try:
        group = asg_obs.describe_auto_scaling_groups(
            AutoScalingGroupNames=[ASG_NAME],
        )["AutoScalingGroups"][0]
        desired = group.get("DesiredCapacity")
        instances = group.get("Instances", [])
        activities = asg_obs.describe_scaling_activities(
            AutoScalingGroupName=ASG_NAME, MaxRecords=3,
        )["Activities"]
    except (ClientError, IndexError) as e:
        desired = "?"
        instances = []
        activities = []
    elapsed = int(360 - (deadline - time.time()))
    print(f"Scale-out progress at t+{elapsed:>3}s:")
    print("-" * 78)
    print(f"  ASG DesiredCapacity: {desired}")
    print(f"  Live instances:")
    for inst in instances:
        print(
            f"    {inst.get('InstanceId', '?'):20}  "
            f"AZ={inst.get('AvailabilityZone', '?'):12}  "
            f"State={inst.get('LifecycleState', '?')}"
        )
    print(f"  Recent scaling activities:")
    for act in activities[:2]:
        print(f"    [{act.get('StatusCode', '?')}] {act.get('Description', '?')}")
        cause = act.get("Cause", "")
        if cause:
            print(f"        Cause: {cause[:160]}")
    if isinstance(desired, int) and desired >= 2:
        final_desired = desired
        LAB_STATE["scale_out_desired"] = desired
        print()
        print(f"Scale-out landed: DesiredCapacity = {desired}.")
        break
    time.sleep(15)
else:
    print()
    print("Timed out waiting for scale-out. Likely causes:")
    print("  - Target-tracking policy not attached (rerun Task 3).")
    print("  - Synthetic load was too light to cross 50 RCPT per minute.")
    print("  - Target was not healthy long enough to define a per-target rate.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: ASG's first instance is healthy in the target group (i.e.,
# the user-data + HTTP server + ALB health check all worked).

def checkpoint_4(session):
    from labrun_checks import CheckpointResult
    try:
        elbv2_c = boto3.client(
            "elbv2",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        health = elbv2_c.describe_target_health(
            TargetGroupArn=LAB_STATE["target_group_arn"],
        )["TargetHealthDescriptions"]
        healthy = [h for h in health if (h.get("TargetHealth") or {}).get("State") == "healthy"]
        if not healthy:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "No healthy targets in the lab target group. The instance "
                    "may still be coming up, or user-data did not start the "
                    "HTTP server (check the EC2 console system log)."
                ),
            )
        return CheckpointResult(status="pass")
    except ClientError as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

**Wallet check.** No new line items from the load helper itself; the synthetic GETs are minuscule. The ASG just launched a second instance, so EC2 burn ticked up to about 1 cent for the session. Running total roughly 4 cents.

## Task 5: Confirm the scale-out reached desired >= 2 and capture the alarm-fire cause

Checkpoint 5 is the final assertion: `DesiredCapacity >= 2` AND the most recent scaling activity carries a `Cause` string that names the target-tracking alarm fire. The observe cell above already prints the cause; this checkpoint pins it down for the lab record.

In [ ]:
# NBVAL_SKIP
# Task 5: verify the scale-out completed and capture the activity cause.

asg = boto3.client(
    "autoscaling",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# YOUR CODE: call asg.describe_scaling_activities(AutoScalingGroupName=ASG_NAME,
# MaxRecords=5) and print each activity's StatusCode, Description, and Cause.
# Capture the Cause string of the most recent SUCCESSFUL scale-out activity in
# a local variable scale_out_cause; the checkpoint will read it from the
# describe call directly so storing it here is for the student's reference.

In [ ]:
# NBVAL_SKIP
# Observe: take a final snapshot of scaling activities so the student sees
# the alarm-fire cause one last time before Checkpoint 5 runs.

asg_obs = boto3.client(
    "autoscaling",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

clear_output(wait=True)
try:
    activities = asg_obs.describe_scaling_activities(
        AutoScalingGroupName=ASG_NAME, MaxRecords=5,
    )["Activities"]
except ClientError as e:
    activities = []
    print(f"Error reading scaling activities: {e}")

print("Scaling activities (most recent first):")
print("=" * 78)
for act in activities:
    print(f"  [{act.get('StatusCode', '?')}] {act.get('Description', '?')}")
    cause = act.get("Cause", "")
    if cause:
        print(f"    Cause: {cause}")
    print()

In [ ]:
# NBVAL_SKIP
# Checkpoint 5: DesiredCapacity >= 2 AND a scaling activity describes the
# alarm-fire cause.

def checkpoint_5(session):
    from labrun_checks import CheckpointResult
    try:
        asg_c = boto3.client(
            "autoscaling",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        group = asg_c.describe_auto_scaling_groups(
            AutoScalingGroupNames=[ASG_NAME],
        )["AutoScalingGroups"][0]
        desired = group.get("DesiredCapacity")
        if not isinstance(desired, int) or desired < 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"ASG DesiredCapacity is {desired}, expected >= 2 after the "
                    f"synthetic load drove RequestCountPerTarget above the threshold."
                ),
            )
        activities = asg_c.describe_scaling_activities(
            AutoScalingGroupName=ASG_NAME, MaxRecords=10,
        )["Activities"]
        scale_out = next(
            (
                a
                for a in activities
                if "Launching a new EC2 instance" in (a.get("Description") or "")
                or "Launching a new EC2 instance" in (a.get("Cause") or "")
            ),
            None,
        )
        if scale_out is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "No scale-out launch activity found in describe_scaling_activities. "
                    "The ASG sizing changed but no launch is recorded; that is unusual."
                ),
            )
        return CheckpointResult(status="pass")
    except ClientError as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(5, checkpoint_5)

<details><summary>Hint 1 (nudge)</summary>

One boto3 call: `describe_scaling_activities`. The recent activities show what the ASG just did and why. The `Cause` field of each activity is a human-readable description of the CloudWatch alarm that triggered the policy.

</details>

<details><summary>Hint 2 (direction)</summary>

`autoscaling.describe_scaling_activities(AutoScalingGroupName=ASG_NAME, MaxRecords=5)` returns `Activities`. Each activity has `StatusCode`, `Description`, `Cause`. A successful scale-out has `StatusCode="Successful"` and a `Description` that starts with `"Launching a new EC2 instance"`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
activities = asg.describe_scaling_activities(
    AutoScalingGroupName=ASG_NAME, MaxRecords=5,
)["Activities"]
for act in activities:
    print(f"[{act['StatusCode']}] {act['Description']}")
    if act.get("Cause"):
        print(f"    Cause: {act['Cause']}")

scale_out_cause = next(
    (a["Cause"] for a in activities if "Launching" in a.get("Description", "")),
    "",
)
```

</details>

**Wallet check.** No new line items. Total session burn lands around 5 to 10 cents depending on how long the second (and possibly third) instance ran. ALB hours plus 2-3 t4g.nano hours are the only meaningful lines.

## Cleanup

The cleanup adapter handles the ASG cascade automatically:

1. Scale the ASG to 0 (Min/Max/Desired) and wait up to 5 minutes for instances to terminate.
2. Delete the ASG.
3. Delete the listener.
4. Delete the ALB (waits until the ENIs detach).
5. Delete the target group.
6. Delete the launch template.
7. Delete both security groups (retries on `DependencyViolation` for up to 5 minutes per group).
8. Delete the instance profile + IAM role.

Total cleanup wall-clock is 5-10 minutes.

In [ ]:
# NBVAL_SKIP
# Cleanup. Critical-first reverse-creation order with the ASG cascade
# pattern from RESOURCE-SAFETY-SPEC Section 3.7.

import sys

_rebuild_manifest()
result = run_cleanup(CLEANUP_MANIFEST)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_types = {"autoscaling_group", "autoscaling_group_scale_to_zero", "elbv2_load_balancer"}
critical_total = sum(1 for r in CLEANUP_MANIFEST if r.type in critical_types)
critical_destroyed = sum(
    1
    for r in CLEANUP_MANIFEST
    if r.type in critical_types and r.id not in failed_ids
)
standard_total = len(CLEANUP_MANIFEST) - critical_total
standard_destroyed = standard_total - sum(
    1
    for rid in failed_ids
    for r in CLEANUP_MANIFEST
    if r.id == rid and r.type not in critical_types
)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: approximately $0.05 to $0.20.** ALB hours dominate at $0.0225/hour; EC2 t4g.nano instances are sub-cent per hour each. The cleanup cell above just terminated everything; the meter stopped the moment that cell printed `Cleanup complete`.

## Reflection

These are not graded. They are for you.

1. Walk through what AWS does in the 60 seconds after the ALB request-count-per-target metric crosses the target-tracking threshold. List the steps: CloudWatch alarm fires, ASG receives the alarm via the policy, ASG launches a new instance from the launch template, the ALB target group registers the new instance, the health check passes, the new target starts taking traffic. Where does the scaling-cooldown window apply in that sequence?

2. Your team is choosing between target-tracking scaling (set a target CPU or request count, AWS auto-creates the alarm and the policy) and step scaling (you define explicit alarm thresholds and step sizes). For a workload with predictable diurnal patterns versus a workload with unpredictable bursts, which is the right call for each, and why?

## Exam-Style Practice

**Question 1 (Domain 2, ASG scaling policy choice for traffic spikes):**

A team runs a public web app behind an ALB and an ASG. Traffic is steady-state at ~100 requests per second during the day, with unpredictable HackerNews-driven spikes that can reach 10x baseline for 10-30 minutes. The team wants to absorb the spike automatically and scale back down within an hour of the spike ending. Which scaling policy fits best?

A. Manual scaling: a human watches the metric and runs `update_auto_scaling_group --desired-capacity` when a spike is detected.

B. Scheduled scaling: pre-schedule scale-out at common HackerNews-busy hours, scale-in after.

C. Target-tracking scaling on `ALBRequestCountPerTarget`: set a target requests-per-instance and let AWS create the alarms and adjust desired capacity automatically.

D. Predictive scaling that uses Forecast to pre-warm capacity for HackerNews spikes before they occur.

<details><summary>Show answer</summary>

**Correct: C.**

- A is wrong: manual scaling is the antipattern the lab exists to replace. A human cannot react in seconds to a HackerNews spike.
- B is wrong: HackerNews spikes are unpredictable in timing; scheduled scaling fits diurnal patterns (e.g., "every morning at 9 AM") not unpredictable bursts.
- C is correct: target-tracking on `ALBRequestCountPerTarget` is the AWS-recommended policy for unpredictable spikes. You declare a target requests-per-instance and AWS handles the alarm logic and capacity adjustments. It scales out fast and scales in gracefully with the configured cooldown.
- D is partially right but predictive scaling needs historical patterns to forecast. HackerNews spikes are not patterned, so predictive scaling has nothing to learn from.

</details>

**Question 2 (Domain 2, high-availability across AZs):**

You are designing an ASG-plus-ALB pattern for high availability. The team's RTO is "no perceptible outage if a single AZ goes dark." The application is stateless behind the ALB. Which configuration achieves the RTO?

A. ASG with min=1, max=3, desired=1 in a single AZ. ALB in the same single AZ.

B. ASG with min=2, max=6, desired=2 across two AZs. ALB across two AZs.

C. ASG with min=1, max=3, desired=1 across two AZs. ALB across two AZs.

D. ASG with min=2, max=6, desired=2 in a single AZ. ALB across two AZs.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: single AZ means a single AZ failure takes the entire workload down. RTO unsatisfied.
- B is correct: ASG with min=2 across two AZs guarantees at least one healthy instance in each AZ at all times. If one AZ goes dark, the surviving AZ keeps serving traffic via the ALB while the ASG launches replacement capacity in the healthy AZ. Min=2 (one per AZ) is the explicit pattern for "no perceptible outage."
- C is wrong: min=1 means one instance total. If that instance lives in the failed AZ, the ASG has to launch a new instance in the surviving AZ, which is a 2-3 minute window where the ALB has no healthy targets. RTO unsatisfied.
- D is wrong: ALB across two AZs is correct, but the ASG in a single AZ means all instances live in that AZ. When the AZ goes dark, the ALB has no healthy targets in either AZ. RTO unsatisfied.

</details>